In [21]:
import os

In [22]:
%pwd

'C:\\Users\\Nitin\\TEXT-SUMMARISER'

In [23]:
os.chdir("../")

In [24]:
os.chdir("../")

In [25]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: Path

In [26]:
import os
os.chdir("C:\\Users\\Nitin\\TEXT-SUMMARISER")
print(os.getcwd())

C:\Users\Nitin\TEXT-SUMMARISER


In [27]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [28]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            tokenizer_name = config.tokenizer_name
        )

        return data_transformation_config

In [29]:
import os
from textSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

In [32]:
import os
os.chdir("C:\\Users\\Nitin\\TEXT-SUMMARISER")

In [33]:
import importlib
import textSummarizer.entity as e
importlib.reload(e)
from textSummarizer.entity import DataTransformationConfig
print("SUCCESS")

SUCCESS


In [40]:
import os
from transformers import AutoTokenizer
from datasets import load_from_disk
from textSummarizer.entity import DataTransformationConfig


class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_path)

    def convert_examples_to_features(self, example_batch):
        input_encodings = self.tokenizer(
            example_batch['dialogue'],
            max_length=1024,
            truncation=True
        )

        target_encodings = self.tokenizer(
            text_target=example_batch['summary'],  # ← fixed
            max_length=128,
            truncation=True
        )

        return {
            'input_ids': input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
        }

    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)
        dataset_samsum_pt = dataset_samsum.map(
            self.convert_examples_to_features,
            batched=True
        )
        dataset_samsum_pt.save_to_disk(
            os.path.join(self.config.root_dir, "samsum_dataset")
        )

In [43]:
import os
os.chdir("C:\\Users\\Nitin\\TEXT-SUMMARISER")

from textSummarizer.config.configuration import ConfigurationManager
from textSummarizer.components.data_transformation import DataTransformation

In [44]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e

[2026-03-22 15:46:50,123: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-22 15:46:50,182: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-22 15:46:50,192: INFO: common: created directory at: artifacts]
[2026-03-22 15:46:50,195: INFO: common: created directory at: artifacts/data_transformation]
[2026-03-22 15:46:51,245: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-03-22 15:46:51,318: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-03-22 15:46:51,716: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-03-22 15:46:51,790: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-c

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 26364.88 examples/s]


[2026-03-22 15:47:33,863: INFO: data_transformation: Data transformation completed!]
